## Monte Carlo Methods with Taxi-v3
Complete all tasks and include answers to the discussion questions in markdown cells.

### Monte Carlo Methods Overview

Monte Carlo methods estimate the value of states by sampling episodes of interaction with the environment. The key concept is the **return**, $G_t$, which is the discounted sum of rewards:
$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^\infty \gamma^k R_{t+k+1}
$$
Here:
- $R_{t+1}$ is the reward received at time step \(t+1\),
- $\gamma$ is the discount factor (\(0 \leq \gamma \leq 1\)).


In [3]:
# Install OpenAI Gym
!pip install gym

# Import libraries
import gym
import numpy as np
import random
import matplotlib.pyplot as plt



[notice] A new release of pip is available: 23.3.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### **1. Environment Setup**

**Task 1: Setup the Taxi-v3 Environment**
1. Initialize the `Taxi-v3` environment.
2. Print the state space and action space dimensions.
3. Render the initial state of the environment.


In [5]:
# Initialize Taxi-v3 environment
env = gym.make("Taxi-v3")

# Display state and action spaces
print(f"State space: {env.observation_space.n}")
print(f"Action space: {env.action_space.n}")

# Render the initial environment state
state = env.reset()
# env.render()

State space: 500
Action space: 6


### **2. First-Visit Monte Carlo Implementation**

### First-Visit Monte Carlo

In **First-Visit Monte Carlo**, we update the value of a state only the **first time** it is visited in an episode. The value \(V(s)\) of a state \(s\) is calculated as:
$$
V(s) \leftarrow \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_t^{(i)}
$$
Where:
- $N(s)$ is the number of first visits to \(s\).
- $G_t^{(i)}$ is the return observed for \(s\) in the \(i\)-th episode.


**Task 2: Implement First-Visit Monte Carlo**

1. Write a function `first_visit_mc` that:
   - Generates episodes by interacting with the Taxi-v3 environment.
   - Calculates the return for each state encountered in the episode. 
   $ G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^\infty \gamma^k R_{t+k+1}$

   - Updates the value of each state only the **first time** it is visited. $V(s) \leftarrow \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_t^{(i)}
$
2. Use ε-greedy exploration during episode generation.
3. Return the final value table after processing all episodes.


In [21]:
def first_visit_mc(env, episodes, gamma=0.99, epsilon=0.1):
    """
    First-Visit Monte Carlo for Taxi-v3.
    
    Args:
        env: The Taxi-v3 environment.
        episodes: Number of episodes to train on.
        gamma: Discount factor for future rewards.
        epsilon: Exploration probability (ε-greedy).
        
    Returns:
        value_table: Estimated state-value function.
    """
    value_table = np.zeros(env.observation_space.n)
    returns = {state: [] for state in range(env.observation_space.n)}
    
    for _ in range(episodes):
        state = env.reset()
        episode = []

        # Generate an episode
        while True:
            # Choose an action using ε-greedy policy
            if random.uniform(0, 1) < epsilon:
                action = env.action_space.sample()  # Explore
            else:
                action = np.argmax(value_table)  # Exploit
            
            next_state, reward, done, info, _ = env.step(action)
            episode.append((state, action, reward))
            
            if done:
                break
            state = next_state

        # Process the episode for first-visit MC
        G = 0
        visited_states = set()
        for state, _, reward in reversed(episode):
            G = reward + gamma * G
            if state not in visited_states:
                visited_states.add(state)
                returns[state].append(G)
                value_table[state] = np.mean(returns[state])

    return value_table

#### **3. Every-Visit Monte Carlo Implementation**

### Every-Visit Monte Carlo

In **Every-Visit Monte Carlo**, we update the value of a state every time it is visited in an episode. The value \(V(s)\) is calculated as:
$$
V(s) \leftarrow \frac{1}{N_{\text{total}}(s)} \sum_{i=1}^{N_{\text{total}}(s)} G_t^{(i)}
$$
Where:
- $N_{\text{total}}(s)$ is the total number of times \(s\) is visited across all episodes.
- $G_t^{(i)}$ is the return observed for \(s\) in the \(i\)-th episode.


### Task 3: Implement Every-Visit Monte Carlo

1. Write a function `every_visit_mc` that:
   - Processes episodes generated in the Taxi-v3 environment.
   - Updates the value of each state every time it is visited in an episode. $V(s) \leftarrow \frac{1}{N_{\text{total}}(s)} \sum_{i=1}^{N_{\text{total}}(s)} G_t^{(i)}
$
2. Use the same ε-greedy exploration as in First-Visit MC.
3. Return the final value table after processing all episodes.


In [22]:
def every_visit_mc(env, episodes, gamma=0.99, epsilon=0.1):
    """
    Every-Visit Monte Carlo for Taxi-v3.
    
    Args:
        env: The Taxi-v3 environment.
        episodes: Number of episodes to train on.
        gamma: Discount factor for future rewards.
        epsilon: Exploration probability (ε-greedy).
        
    Returns:
        value_table: Estimated state-value function.
    """
    value_table = np.zeros(env.observation_space.n)
    returns = {state: [] for state in range(env.observation_space.n)}
    
    for _ in range(episodes):
        state = env.reset()
        episode = []

        # Generate an episode
        while True:
            # Choose an action using ε-greedy policy
            if random.uniform(0, 1) < epsilon:
                action = env.action_space.sample()  # Explore
            else:
                action = np.argmax(value_table)  # Exploit
            
            next_state, reward, done,info, _ = env.step(action)
            episode.append((state, action, reward))
            print(f'Episode #{_} Reward: {reward}')
            
            if done:
                break
            state = next_state

        # Process the episode for every-visit MC
        G = 0
        for state, _, reward in reversed(episode):
            G = reward + gamma * G
            returns[state].append(G)
            value_table[state] = np.mean(returns[state])

    return value_table

#### **4. Policy Evaluation**

### Policy Evaluation

The agent chooses actions using a derived policy. For any state \(s\), the action \(a\) with the highest expected return \(Q(s, a)\) is selected:
$$
a = \arg\max_{a'} Q(s, a')
$$


### Task 4: Evaluate the Policy

1. Write a function `evaluate_policy` that:
   - Simulates several episodes using the learned value table to derive a policy.
   - Executes the policy and collects total rewards over the episodes.
   $a = \arg\max_{a'} Q(s, a')
$
2. Compute the **average reward per episode** and return it.


In [23]:
def evaluate_policy(env, value_table, episodes=100):
    """
    Evaluate the performance of a policy derived from a value table.
    
    Args:
        env: The Taxi-v3 environment.
        value_table: The state-value function.
        episodes: Number of episodes to evaluate over.
        
    Returns:
        average_reward: Average reward per episode.
    """
    total_reward = 0
    
    for _ in range(episodes):
        state = env.reset()
        episode_reward = 0
        
        while True:
            # Choose the action with the highest value
            action = np.argmax([value_table[state] for _ in range(env.action_space.n)])
            state, reward, done, info, _ = env.step(action)
            episode_reward += reward
            print(f'Episode #{_} Reward: {episode_reward}')
            if done:
                break
        
        total_reward += episode_reward
    
    return total_reward / episodes

#### **5. Visualization and Comparison**

### Task 5: Visualize and Compare Value Tables

1. Write a function `plot_value_comparison` to:
   - Plot value estimates from **First-Visit MC** and **Every-Visit MC** side by side for selected states.
2. Use the first 20 states for comparison.
3. Analyze the differences in value estimates for the two methods.


In [24]:
def plot_value_comparison(fv_values, ev_values, title="Value Comparison"):
    """
    Compare and visualize value estimates from FV-MC and EV-MC.
    
    Args:
        fv_values: Value estimates from First-Visit MC.
        ev_values: Value estimates from Every-Visit MC.
        title: Title for the plot.
    """
    indices = np.arange(len(fv_values))
    
    plt.figure(figsize=(12, 6))
    plt.bar(indices - 0.2, fv_values, width=0.4, label="First-Visit MC", color="blue")
    plt.bar(indices + 0.2, ev_values, width=0.4, label="Every-Visit MC", color="orange")
    plt.xlabel("State Index")
    plt.ylabel("Value Estimate")
    plt.title(title)
    plt.legend()
    plt.show()

#### **6. Main Execution**

In [25]:
# Training parameters
episodes = 5000
gamma = 0.99
epsilon = 0.1

### Task 6: Train the Agent Using First-Visit Monte Carlo

1. Train the agent using your `first_visit_mc` implementation for 5000 episodes.
2. Save the resulting value table.
3. Print the value estimates for a few sample states.


In [ ]:
# Train using First-Visit MC
fv_value_table = first_visit_mc(env, episodes, gamma, epsilon)

### Task 7: Train the Agent Using Every-Visit Monte Carlo

1. Train the agent using your `every_visit_mc` implementation for 5000 episodes.
2. Save the resulting value table.
3. Print the value estimates for a few sample states.


In [ ]:
# Train using Every-Visit MC
ev_value_table = every_visit_mc(env, episodes, gamma, epsilon)

### Task 8: Compare the Performance of FV-MC and EV-MC

1. Use your `plot_value_comparison` function to compare value estimates.
2. Evaluate the average reward of the policies derived from FV-MC and EV-MC.
3. Answer the following questions:
   - Which method produces higher average rewards?
   - Are there significant differences in the value estimates? Why or why not?
   - How do starting states influence the performance of the two methods?


### Comparison of First-Visit and Every-Visit Monte Carlo

Both methods estimate the state-value function $V(s)$, but they differ in how they process returns:
- **First-Visit Monte Carlo**: Updates the value of a state only for its first occurrence in an episode.
- **Every-Visit Monte Carlo**: Updates the value of a state every time it is visited in an episode.

Visualize the results and analyze their differences.


In [ ]:
# Compare value tables for the first 20 states
plot_value_comparison(fv_value_table[:20], ev_value_table[:20], title="Value Comparison for First 20 States")

# Evaluate the performance of both methods
fv_performance = evaluate_policy(env, fv_value_table)
ev_performance = evaluate_policy(env, ev_value_table)

print(f"First-Visit MC Average Reward: {fv_performance}")
print(f"Every-Visit MC Average Reward: {ev_performance}")

### Exploration with \(\epsilon\)-Greedy Policy

To ensure sufficient exploration, actions are chosen using an \(\epsilon\)-greedy policy:
$$
\pi(a|s) = 
\begin{cases} 
1 - \epsilon + \frac{\epsilon}{|\mathcal{A}|} & \text{if } a = \arg\max_{a'} Q(s, a') \\
\frac{\epsilon}{|\mathcal{A}|} & \text{otherwise}
\end{cases}
$$
Where:
- $|\mathcal{A}|$ is the total number of actions.
- $\epsilon$ is the probability of exploring (choosing a random action).
